# fGPU Fractional GPU 공유 실험 — 종합 분석 노트북

이 노트북은 두 세션(A, B)의 `session_result_*.csv` 파일을 glob으로 수집하고  
시나리오별 throughput, makespan, speedup 을 계산해 그래프와 요약을 생성합니다.

**호스트에서 실행**: GPU 불필요. pandas + matplotlib 만 사용합니다.

전제:
- `fgpu_infer.ipynb` 를 각 세션에서 최소 solo_A, solo_B, concurrent(A+B) 실행 완료.
- 결과 파일 위치: `<repo>/data/sessions/<session_id>/session_result_<scenario>.csv`


In [ ]:
# ══════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════
import os
import pathlib

# data/sessions 디렉토리 (호스트 경로, 절대경로 override 가능)
_default = pathlib.Path(__file__).resolve().parent.parent / "data" / "sessions" if "__file__" in dir() else pathlib.Path("/home/user/GpuCluster/data/sessions")
SESSIONS_DIR = str(os.environ.get("FGPU_SESSIONS_DIR", _default))

# 그래프 저장 디렉토리 (노트북 옆에 생성)
OUTPUT_DIR = pathlib.Path(SESSIONS_DIR).parent.parent / "notebooks" / "analysis_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# speedup 판정 임계값
SPEEDUP_PASS_THRESHOLD = 1.2

print(f"SESSIONS_DIR : {SESSIONS_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")


In [ ]:
# ══════════════════════════════════════════════════════
# 설치 (호스트 venv 포함, 멱등)
# ══════════════════════════════════════════════════════
import sys, subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip install 실패:", result.stderr[:300])
else:
    print("pandas, matplotlib 설치/확인 완료")


In [ ]:
# ══════════════════════════════════════════════════════
# session_result_*.csv 수집
# ══════════════════════════════════════════════════════
import glob
import pathlib
import pandas as pd

sessions_path = pathlib.Path(SESSIONS_DIR)
pattern = str(sessions_path / "**" / "session_result_*.csv")
csv_files = sorted(glob.glob(pattern, recursive=True))

print(f"발견된 CSV 파일 ({len(csv_files)}개):")
for f in csv_files:
    print(f"  {f}")

if not csv_files:
    print()
    print("[경고] CSV 파일 없음.")
    print(f"       SESSIONS_DIR='{SESSIONS_DIR}' 를 확인하세요.")
    print("       또는 FGPU_SESSIONS_DIR 환경변수로 경로를 지정하세요.")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        # 파일명에서 session_id 추출 (data/sessions/<sid>/session_result_*.csv)
        parts = pathlib.Path(f).parts
        # sessions/<sid>/ 패턴에서 sid 위치
        try:
            sid_idx = [i for i, p in enumerate(parts) if p == "sessions"][-1] + 1
            df["session_id"] = parts[sid_idx] if sid_idx < len(parts) else "unknown"
        except Exception:
            df["session_id"] = "unknown"
        dfs.append(df)
    except Exception as e:
        print(f"  [오류] {f}: {e}")

if dfs:
    all_data = pd.concat(dfs, ignore_index=True)
    print(f"\n전체 행 수: {len(all_data)}")
    print(f"컬럼: {list(all_data.columns)}")
    print(all_data.head(3))
else:
    all_data = pd.DataFrame()
    print("데이터 없음. 이후 셀은 빈 결과를 출력합니다.")


In [ ]:
# ══════════════════════════════════════════════════════
# session_meta_*.json 수집 (요약 지표 포함)
# ══════════════════════════════════════════════════════
import glob, json, pathlib

meta_pattern = str(pathlib.Path(SESSIONS_DIR) / "**" / "session_meta_*.json")
meta_files = sorted(glob.glob(meta_pattern, recursive=True))

print(f"메타 JSON 파일 ({len(meta_files)}개):")
metas = []
for f in meta_files:
    try:
        with open(f) as jf:
            m = json.load(jf)
        # session_id 추가
        parts = pathlib.Path(f).parts
        try:
            sid_idx = [i for i, p in enumerate(parts) if p == "sessions"][-1] + 1
            m["session_id"] = parts[sid_idx] if sid_idx < len(parts) else "unknown"
        except Exception:
            m["session_id"] = "unknown"
        metas.append(m)
        oom_str = "OOM" if m.get("oom") else "OK"
        print(f"  {m.get('label','?')}  scenario={m.get('scenario','?')}"
              f"  ratio={m.get('ratio','?')}  {oom_str}"
              f"  makespan={m.get('makespan_s', 0):.1f}s"
              f"  tps={m.get('mean_tokens_per_s', 0):.1f}")
    except Exception as e:
        print(f"  [오류] {f}: {e}")

if not metas:
    print("[경고] 메타 파일 없음.")


In [ ]:
# ══════════════════════════════════════════════════════
# 시나리오별 집계 & 이득 지표 계산
# ══════════════════════════════════════════════════════
import statistics

# solo 기준선 추출
solo_stats = {}   # key: label ("A"/"B") → meta dict
for m in metas:
    if m.get("scenario") == "solo":
        solo_stats[m.get("label")] = m

# concurrent / seq 추출 (label 기준 A, B 쌍 구성)
conc_by_scenario: dict[str, dict] = {}  # scenario → {label: meta}
seq_by_scenario: dict[str, dict] = {}

for m in metas:
    sc = m.get("scenario", "")
    lbl = m.get("label", "?")
    if sc not in ("solo",):
        if "conc" in sc or sc == "concurrent":
            conc_by_scenario.setdefault(sc, {})[lbl] = m
        elif "seq" in sc or sc == "seq":
            seq_by_scenario.setdefault(sc, {})[lbl] = m

print("=" * 60)
print("solo 기준선:")
for lbl, m in sorted(solo_stats.items()):
    print(f"  {lbl}: tps={m.get('mean_tokens_per_s',0):.2f} tok/s"
          f"  makespan={m.get('makespan_s',0):.2f}s"
          f"  peak_mem={m.get('peak_mem_alloc_mib',0):.0f}MiB")

def _calc_speedup(metas_pair: dict, solo_a, solo_b):
    """
    metas_pair: {"A": meta_A, "B": meta_B}  (concurrent)
    solo_a, solo_b: solo 메타
    반환: (speedup, e_share, contention_penalty, makespan_conc, makespan_seq)
    """
    if "A" not in metas_pair or "B" not in metas_pair:
        return None, None, None, None, None
    ma, mb = metas_pair["A"], metas_pair["B"]
    if solo_a is None or solo_b is None:
        return None, None, None, None, None

    t_start = min(ma.get("t_start_epoch", 0), mb.get("t_start_epoch", 0))
    t_end   = max(ma.get("t_end_epoch",   0), mb.get("t_end_epoch",   0))
    makespan_conc = t_end - t_start if t_end > t_start else 0

    wall_solo_a = solo_a.get("makespan_s", 0)
    wall_solo_b = solo_b.get("makespan_s", 0)
    makespan_seq = wall_solo_a + wall_solo_b

    speedup = (makespan_seq / makespan_conc) if makespan_conc > 0 else 0
    e_share = (wall_solo_a + wall_solo_b) / makespan_conc if makespan_conc > 0 else 0

    tps_solo_a = solo_a.get("mean_tokens_per_s", 0)
    tps_solo_b = solo_b.get("mean_tokens_per_s", 0)
    tps_conc_a = ma.get("mean_tokens_per_s", 0)
    tps_conc_b = mb.get("mean_tokens_per_s", 0)
    denom = tps_solo_a + tps_solo_b
    contention_penalty = (
        1 - (tps_conc_a + tps_conc_b) / denom
        if denom > 0 else 0
    )

    return speedup, e_share, contention_penalty, makespan_conc, makespan_seq


print()
print("concurrent 시나리오 이득 지표:")
_results_table = []
for sc, pair in sorted(conc_by_scenario.items()):
    lbl_a = pair.get("A", {}).get("label", "A")
    lbl_b = pair.get("B", {}).get("label", "B")
    sol_a = solo_stats.get("A")
    sol_b = solo_stats.get("B")
    spd, esh, cpen, mk_conc, mk_seq = _calc_speedup(pair, sol_a, sol_b)
    verdict = "N/A"
    if spd is not None:
        verdict = "PASS(이득)" if spd >= SPEEDUP_PASS_THRESHOLD else "FAIL(이득미달)"
    print(f"  {sc}:")
    if spd is not None:
        print(f"    speedup           = {spd:.3f}  → {verdict}")
        print(f"    E_share           = {esh:.3f}  (>1=이득)")
        print(f"    contention_penalty= {cpen:.3f}  (낮을수록 경합 적음)")
        print(f"    makespan_conc     = {mk_conc:.2f}s")
        print(f"    makespan_seq      = {mk_seq:.2f}s")
    else:
        print(f"    데이터 부족 (solo A/B 또는 conc A/B 쌍이 없음)")
    _results_table.append({
        "scenario": sc, "speedup": spd, "e_share": esh,
        "contention_penalty": cpen, "makespan_conc_s": mk_conc,
        "makespan_seq_s": mk_seq, "verdict": verdict,
    })


In [ ]:
# ══════════════════════════════════════════════════════
# runs.csv 저장
# ══════════════════════════════════════════════════════
import csv, pathlib

runs_csv = OUTPUT_DIR / "runs.csv"
fieldnames = [
    "scenario", "label_a", "label_b",
    "ratio_a", "ratio_b",
    "tps_solo_a", "tps_solo_b",
    "tps_conc_a", "tps_conc_b",
    "peak_mem_mib_a", "peak_mem_mib_b",
    "makespan_seq_s", "makespan_conc_s",
    "speedup", "e_share", "contention_penalty",
    "verdict",
]

with open(runs_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in _results_table:
        sc = row["scenario"]
        pair = conc_by_scenario.get(sc, {})
        ma = pair.get("A", {})
        mb = pair.get("B", {})
        sol_a = solo_stats.get("A", {})
        sol_b = solo_stats.get("B", {})
        writer.writerow({
            "scenario":    sc,
            "label_a":     ma.get("label", "A"),
            "label_b":     mb.get("label", "B"),
            "ratio_a":     ma.get("ratio", ""),
            "ratio_b":     mb.get("ratio", ""),
            "tps_solo_a":  sol_a.get("mean_tokens_per_s", ""),
            "tps_solo_b":  sol_b.get("mean_tokens_per_s", ""),
            "tps_conc_a":  ma.get("mean_tokens_per_s", ""),
            "tps_conc_b":  mb.get("mean_tokens_per_s", ""),
            "peak_mem_mib_a": ma.get("peak_mem_alloc_mib", ""),
            "peak_mem_mib_b": mb.get("peak_mem_alloc_mib", ""),
            "makespan_seq_s":  row.get("makespan_seq_s", ""),
            "makespan_conc_s": row.get("makespan_conc_s", ""),
            "speedup":     row.get("speedup", ""),
            "e_share":     row.get("e_share", ""),
            "contention_penalty": row.get("contention_penalty", ""),
            "verdict":     row.get("verdict", ""),
        })

print(f"runs.csv 저장: {runs_csv}")


In [ ]:
# ══════════════════════════════════════════════════════
# 그래프 (a) speedup 막대  (b) tokens/sec 비교  (c) makespan 비교
# ══════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")   # 서버 환경 (display 없음) 대응
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── (a) speedup 막대 ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("fGPU Fractional GPU 공유 실험 — 시나리오별 비교", fontsize=13)

ax_spd = axes[0]
sc_names = [r["scenario"] for r in _results_table if r.get("speedup") is not None]
speedups  = [r["speedup"]  for r in _results_table if r.get("speedup") is not None]
colors_spd = ["steelblue" if s >= SPEEDUP_PASS_THRESHOLD else "salmon" for s in speedups]

if sc_names:
    bars = ax_spd.bar(range(len(sc_names)), speedups, color=colors_spd)
    ax_spd.axhline(y=SPEEDUP_PASS_THRESHOLD, color="green", linestyle="--",
                   linewidth=1.5, label=f"Pass threshold ({SPEEDUP_PASS_THRESHOLD})")
    ax_spd.axhline(y=1.0, color="gray", linestyle=":", linewidth=1, label="baseline=1.0")
    ax_spd.set_xticks(range(len(sc_names)))
    ax_spd.set_xticklabels(sc_names, rotation=20, ha="right", fontsize=8)
    ax_spd.set_ylabel("speedup (makespan_seq / makespan_conc)")
    ax_spd.set_title("(a) Speedup by scenario")
    ax_spd.legend(fontsize=7)
    for i, (s, b) in enumerate(zip(speedups, bars)):
        ax_spd.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                    f"{s:.2f}", ha="center", va="bottom", fontsize=8)
else:
    ax_spd.text(0.5, 0.5, "데이터 없음", ha="center", va="center",
                transform=ax_spd.transAxes)
    ax_spd.set_title("(a) Speedup by scenario")

# ── (b) tokens/sec 비교 grouped bar ───────────────────
ax_tps = axes[1]
if sc_names and solo_stats:
    solo_a_tps = solo_stats.get("A", {}).get("mean_tokens_per_s", 0)
    solo_b_tps = solo_stats.get("B", {}).get("mean_tokens_per_s", 0)
    x_vals = [solo_a_tps, solo_b_tps]
    x_labels = ["solo_A", "solo_B"]
    colors_t = ["dodgerblue", "orange"]
    for sc in [r["scenario"] for r in _results_table]:
        pair = conc_by_scenario.get(sc, {})
        x_vals.append(pair.get("A", {}).get("mean_tokens_per_s", 0))
        x_vals.append(pair.get("B", {}).get("mean_tokens_per_s", 0))
        x_labels += [f"A ({sc})", f"B ({sc})"]
        colors_t.extend(["steelblue", "tomato"])
    ax_tps.bar(range(len(x_vals)), x_vals, color=colors_t[:len(x_vals)])
    ax_tps.set_xticks(range(len(x_labels)))
    ax_tps.set_xticklabels(x_labels, fontsize=7, rotation=15, ha="right")
    ax_tps.set_ylabel("mean tokens/sec")
    ax_tps.set_title("(b) Tokens/sec: solo vs concurrent")
    solo_patch = mpatches.Patch(color="dodgerblue", label="solo A")
    conc_a_patch = mpatches.Patch(color="steelblue", label="conc A")
    conc_b_patch = mpatches.Patch(color="tomato", label="conc B")
    ax_tps.legend(handles=[solo_patch, conc_a_patch, conc_b_patch], fontsize=7)
else:
    ax_tps.text(0.5, 0.5, "데이터 없음", ha="center", va="center",
                transform=ax_tps.transAxes)
    ax_tps.set_title("(b) Tokens/sec: solo vs concurrent")

# ── (c) makespan 비교 ─────────────────────────────────
ax_mk = axes[2]
if _results_table:
    r = _results_table[0]   # 첫 시나리오
    sc_name = r["scenario"]
    labels_mk = ["makespan_seq (solo_A+solo_B)", f"makespan_conc ({sc_name})"]
    vals_mk   = [r.get("makespan_seq_s") or 0, r.get("makespan_conc_s") or 0]
    bar_colors = ["slategray", "steelblue"]
    ax_mk.bar(labels_mk, vals_mk, color=bar_colors)
    ax_mk.set_ylabel("seconds")
    title_str = f"(c) Makespan 비교 ({sc_name})"
    ax_mk.set_title(title_str)
    for i, v in enumerate(vals_mk):
        ax_mk.text(i, v + 0.5, f"{v:.1f}s", ha="center", va="bottom", fontsize=9)
    if vals_mk[0] > 0 and vals_mk[1] > 0:
        spd_text = f"speedup={vals_mk[0]/vals_mk[1]:.2f}"
        ax_mk.text(0.5, 0.95, spd_text, ha="center", va="top",
                   transform=ax_mk.transAxes, fontsize=10, color="green")
else:
    ax_mk.text(0.5, 0.5, "데이터 없음", ha="center", va="center",
               transform=ax_mk.transAxes)
    ax_mk.set_title("(c) Makespan 비교")

plt.tight_layout()
fig_path = OUTPUT_DIR / "fgpu_sharing_results.png"
plt.savefig(str(fig_path), dpi=150, bbox_inches="tight")
plt.show()
print(f"그래프 저장: {fig_path}")


In [ ]:
# ══════════════════════════════════════════════════════
# 그래프 (d) GPU util 타임라인 (pynvml 보조 지표)
# t_start_epoch 기준으로 A/B 상대 시간 정렬
# ══════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig2, ax = plt.subplots(figsize=(12, 4))
ax.set_title("GPU Util 타임라인 (pynvml 전체 GPU, 보조 지표\n"
             "두 세션 합산 — per-process 분해 불가)")
ax.set_xlabel("경과 시간 (s, t_start_A 기준)")
ax.set_ylabel("GPU util %")

# 공통 t0: concurrent 시나리오에서 min(t_start_A, t_start_B)
all_conc_metas = []
for sc, pair in conc_by_scenario.items():
    for lbl, m in pair.items():
        all_conc_metas.append(m)

if all_conc_metas:
    t0 = min(m.get("t_start_epoch", 0) for m in all_conc_metas)
    colors_map = {"A": "steelblue", "B": "tomato"}
    plotted = False
    for sc, pair in conc_by_scenario.items():
        for lbl, m in pair.items():
            timeline = m.get("gpu_util_timeline", [])
            if not timeline:
                continue
            xs = [s["ts"] - t0 for s in timeline]
            ys = [s["util_pct"] for s in timeline]
            ax.plot(xs, ys, alpha=0.7,
                    color=colors_map.get(lbl, "gray"),
                    label=f"{lbl} ({sc})")
            plotted = True
    if plotted:
        ax.legend()
        ax.set_ylim(0, 105)
        ax.axhline(80, color="orange", linestyle="--", linewidth=0.8,
                   label="80% (compute-bound 기준)")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "pynvml 데이터 없음\n(PYNVML_OK=False 또는 solo 시나리오만 있음)",
                ha="center", va="center", transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "concurrent 메타 없음",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
fig2_path = OUTPUT_DIR / "gpu_util_timeline.png"
plt.savefig(str(fig2_path), dpi=150, bbox_inches="tight")
plt.show()
print(f"타임라인 그래프 저장: {fig2_path}")
print("[주의] GPU util 은 두 컨테이너 합산 전체 수치입니다.")
print("       개별 세션 분리는 torch.cuda.max_memory_allocated() 를 참조하세요.")


In [ ]:
# ══════════════════════════════════════════════════════
# summary.txt 저장
# ══════════════════════════════════════════════════════
import time as _t_sum

summary_path = OUTPUT_DIR / "summary.txt"

lines = []
lines.append("=" * 60)
lines.append("fGPU Fractional GPU 공유 실험 — 분석 요약")
lines.append(f"생성 시각: {_t_sum.strftime('%Y-%m-%dT%H:%M:%SZ', _t_sum.gmtime())}")
lines.append("=" * 60)
lines.append("")

# solo 기준선
lines.append("[ solo 기준선 ]")
for lbl, m in sorted(solo_stats.items()):
    lines.append(f"  {lbl}: tps={m.get('mean_tokens_per_s',0):.2f} tok/s  "
                 f"makespan={m.get('makespan_s',0):.2f}s  "
                 f"peak_mem={m.get('peak_mem_alloc_mib',0):.0f}MiB")
lines.append("")

# concurrent 결과
lines.append("[ concurrent 시나리오 이득 지표 ]")
for row in _results_table:
    sc = row.get("scenario", "?")
    spd = row.get("speedup")
    esh = row.get("e_share")
    cpen = row.get("contention_penalty")
    mk_c = row.get("makespan_conc_s")
    mk_s = row.get("makespan_seq_s")
    verd = row.get("verdict", "N/A")
    if spd is not None:
        lines.append(f"  {sc}:")
        lines.append(f"    speedup              = {spd:.3f}")
        lines.append(f"    VERDICT              : {verd}")
        lines.append(f"    E_share              = {esh:.3f}")
        lines.append(f"    contention_penalty   = {cpen:.3f}")
        lines.append(f"    makespan_conc        = {mk_c:.2f}s")
        lines.append(f"    makespan_seq         = {mk_s:.2f}s")
    else:
        lines.append(f"  {sc}: 데이터 부족")
    lines.append("")

# 전체 VERDICT
lines.append("=" * 60)
if _results_table:
    any_pass = any(r.get("speedup") is not None and r["speedup"] >= SPEEDUP_PASS_THRESHOLD
                   for r in _results_table)
    all_data_ok = all(r.get("speedup") is not None for r in _results_table)
    if not all_data_ok:
        overall = "INCONCLUSIVE (데이터 부족)"
    elif any_pass:
        overall = "PASS — 최소 1 시나리오에서 speedup ≥ 1.2 (H2 공유 이득 입증)"
    else:
        overall = "FAIL — speedup < 1.2 (H1: 컴퓨트 바운드, 공유 이득 없음)"
    lines.append(f"VERDICT: {overall}")
else:
    lines.append("VERDICT: INCONCLUSIVE (측정 데이터 없음)")
lines.append("=" * 60)
lines.append("")

# 한계 섹션
lines.append("[ 의도된 한계 ]")
limits = [
    "SM 격리 없음: 두 컨테이너가 동일 SM 자유 경쟁 (cooperative wall-clock throttle만)",
    "PYTORCH_NO_CUDA_MEMORY_CACHING=1: KV cache 매 스텝 cudaMalloc/Free → 5~30x 저하",
    "pynvml per-process 불가: 전체 GPU util만 보조 지표 (torch.cuda.memory 주력)",
    "배리어 skew: 수동 동시 Shift+Enter → 수 초 skew 가능 (방법 B 한계)",
    "admission 통과 ≠ 물리 안전: CUDA context ~700MiB×2 가 quota 밖",
    "launch count ≠ device time: cudaLaunchKernel 카운터는 커널 무게 미반영",
]
for lim in limits:
    lines.append(f"  - {lim}")

summary_text = "\n".join(lines)
with open(summary_path, "w") as f:
    f.write(summary_text)

print(summary_text)
print()
print(f"summary.txt 저장: {summary_path}")
